In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_hotel_bookings.csv")

df.shape

(87230, 31)

In [2]:
df.columns.tolist()

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

In [3]:
for col in df.columns:
    print(col)

hotel
is_canceled
lead_time
arrival_date_year
arrival_date_month
arrival_date_week_number
arrival_date_day_of_month
stays_in_weekend_nights
stays_in_week_nights
adults
children
babies
meal
country
market_segment
distribution_channel
is_repeated_guest
previous_cancellations
previous_bookings_not_canceled
reserved_room_type
assigned_room_type
booking_changes
deposit_type
agent
days_in_waiting_list
customer_type
adr
required_car_parking_spaces
total_of_special_requests
reservation_status
reservation_status_date


In [4]:
pd.set_option("display.max_rows", None)

pd.Series(df.columns)

0                              hotel
1                        is_canceled
2                          lead_time
3                  arrival_date_year
4                 arrival_date_month
5           arrival_date_week_number
6          arrival_date_day_of_month
7            stays_in_weekend_nights
8               stays_in_week_nights
9                             adults
10                          children
11                            babies
12                              meal
13                           country
14                    market_segment
15              distribution_channel
16                 is_repeated_guest
17            previous_cancellations
18    previous_bookings_not_canceled
19                reserved_room_type
20                assigned_room_type
21                   booking_changes
22                      deposit_type
23                             agent
24              days_in_waiting_list
25                     customer_type
26                               adr
2

In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

pd.Series(df.columns)

0                              hotel
1                        is_canceled
2                          lead_time
3                  arrival_date_year
4                 arrival_date_month
5           arrival_date_week_number
6          arrival_date_day_of_month
7            stays_in_weekend_nights
8               stays_in_week_nights
9                             adults
10                          children
11                            babies
12                              meal
13                           country
14                    market_segment
15              distribution_channel
16                 is_repeated_guest
17            previous_cancellations
18    previous_bookings_not_canceled
19                reserved_room_type
20                assigned_room_type
21                   booking_changes
22                      deposit_type
23                             agent
24              days_in_waiting_list
25                     customer_type
26                               adr
2

In [6]:
print("\n".join(df.columns))

hotel
is_canceled
lead_time
arrival_date_year
arrival_date_month
arrival_date_week_number
arrival_date_day_of_month
stays_in_weekend_nights
stays_in_week_nights
adults
children
babies
meal
country
market_segment
distribution_channel
is_repeated_guest
previous_cancellations
previous_bookings_not_canceled
reserved_room_type
assigned_room_type
booking_changes
deposit_type
agent
days_in_waiting_list
customer_type
adr
required_car_parking_spaces
total_of_special_requests
reservation_status
reservation_status_date


In [7]:
df_model = df.drop(columns=["reservation_status", "reservation_status_date"])

df_model.shape

(87230, 29)

## Removing clear leakage columns

The columns `reservation_status` and `reservation_status_date` were removed from the modelling dataframe.

The above columns were identified during EDA as strong leakage risks because they contain information related to the final booking outcome.

Removing them helps ensure the later model is trained only on features that would be known before the cancellation outcome occurs.

In [8]:
df_model.columns.tolist()

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests']

In [9]:
X = df_model.drop(columns=["is_canceled"])
y = df_model["is_canceled"]

X.shape, y.shape

((87230, 28), (87230,))

In [10]:
X.dtypes.value_counts()

int64      15
object     10
float64     3
Name: count, dtype: int64

In [11]:
X.select_dtypes(include="object").columns.tolist()

['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type']

## Current modelling setup

The target variable has been separated as:

- `y = is_canceled`

The feature matrix has been separated as:

- `X = all remaining modelling features`

At this stage:
- `X` contains 28 feature columns
- 15 columns are `int64`
- 3 columns are `float64`
- 10 columns are `object`

The categorical feature columns are:
- `hotel`
- `arrival_date_month`
- `meal`
- `country`
- `market_segment`
- `distribution_channel`
- `reserved_room_type`
- `assigned_room_type`
- `deposit_type`
- `customer_type`

The above means encoding will be required before ML models can be trained.

In [12]:
X_encoded = pd.get_dummies(X, drop_first=True)

X_encoded.shape

(87230, 245)

## Encoding categorical features

The feature matrix `X` was one-hot encoded using `pd.get_dummies()` with `drop_first=True`.

This converted the categorical features into numeric dummy variables so they can be used in the ML models.

After encoding:
- the original feature matrix had 28 columns.
- the encoded feature matrix has 245 columns.

The above increase in feature count is expected because categorical variables such as `country`, `market_segment`, and room type create multiple binary columns.

In [13]:
X_encoded.columns[:20]

Index(['lead_time', 'arrival_date_year', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'booking_changes', 'agent',
       'days_in_waiting_list', 'adr', 'required_car_parking_spaces',
       'total_of_special_requests', 'hotel_Resort Hotel',
       'arrival_date_month_August'],
      dtype='object')

In [14]:
X_encoded.to_csv("../data/processed/X_encoded.csv", index=False)
y.to_csv("../data/processed/y.csv", index=False)